# Annotator smoke test -- real IHC TMA

Minimal, standalone check of `barna_utils.annotator.MultiRegionAnnotator` against
IHC's actual stitched TMA OME-TIFF -- not part of any real pipeline, just a place
to click through pan/zoom/draw/select/edit/rename/toggle-channel/contrast/maximize
for real before trusting this in production. Annotations saved here go to
`test_annotations/`, kept separate from any real IHC output.

Needs an interactive backend (`ipympl`) and read access to the TMA file below.
No barna-utils installation required -- the `sys.path.insert` below imports it
directly from source, so this runs in any env that already has the annotator's
dependencies (matplotlib, shapely, geopandas, tifffile>=2026.6, zarr, scikit-image,
imagecodecs) without needing to modify that environment -- e.g. `ihc_stitch`,
which already has all of these.

In [ ]:
%matplotlib widget
import sys
sys.path.insert(0, '/omics/odcf/analysis/OE0146_projects/idh_astro/Barna/Projects/barna-utils/src')

from barna_utils.annotator import MultiRegionAnnotator
from barna_utils.image_sources import ome_tiff_canvas_provider

In [ ]:
TMA_PATH = '/omics/odcf/analysis/OE0146_projects/idh_astro/Barna/Projects/IHC/Data/Raw/TMAs/TMA_merged.ome.tiff'
ANNOTATIONS_DIR = '/omics/odcf/analysis/OE0146_projects/idh_astro/Barna/Projects/barna-utils/examples/test_annotations'

# One color per channel -- adjust to taste once you can see the real image;
# this is just a reasonable starting composite for a 6-channel panel.
CHANNEL_COLORS = {0: 'blue', 1: 'green', 2: 'red', 3: 'cyan', 4: 'magenta', 5: 'yellow'}

provider = ome_tiff_canvas_provider(
    {'tma1': TMA_PATH},
    channels=CHANNEL_COLORS,
    target_max_dim=2048,   # canvas resolution -- raise for more detail, lower for a faster load
)

In [ ]:
ann = MultiRegionAnnotator(
    runs=['tma1'],
    canvas_provider=provider,
    annotations_dir=ANNOTATIONS_DIR,
    px_per_micron=1.0,   # placeholder -- coordinates here are canvas pixels, not real microns
)
ann.display()

---
## Second test: the freshly-merged whole-section OME-TIFF

IHC's own `01_IHC_stitching.ipynb` just finished writing this (5-channel pyramidal
BigTIFF, `Data/Processed/SingleSections/stitched_pyramid.ome.tif`) -- checking it here
means confirming the stitch looks right directly on the cluster, without downloading
a 150GB+ file locally first.

In [ ]:
SECTION_PATH = '/omics/odcf/analysis/OE0146_projects/idh_astro/Barna/Projects/IHC/Data/Processed/SingleSections/stitched_pyramid.ome.tif'
SECTION_ANNOTATIONS_DIR = '/omics/odcf/analysis/OE0146_projects/idh_astro/Barna/Projects/barna-utils/examples/test_annotations_section'

# 5 channels this time -- adjust once you can see which channel is which
SECTION_CHANNEL_COLORS = {0: 'blue', 1: 'green', 2: 'red', 3: 'cyan', 4: 'magenta'}

section_provider = ome_tiff_canvas_provider(
    {'section1': SECTION_PATH},
    channels=SECTION_CHANNEL_COLORS,
    target_max_dim=2048,
)

In [ ]:
ann_section = MultiRegionAnnotator(
    runs=['section1'],
    canvas_provider=section_provider,
    annotations_dir=SECTION_ANNOTATIONS_DIR,
    px_per_micron=1.0,
)
ann_section.display()

## What to check

- **Pan/zoom**: scroll to zoom, middle-click-drag to pan.
- **Draw**: left-click to place vertices, right-click to close the polygon, edit the
  name if needed, Confirm.
- **Select / Edit / Rename / Del sel. / Del last / Undo / Cancel / Reload**: as labeled.
- **Channel toggle row** (small colored buttons above the image): click one to hide/show
  that channel in the composite.
- **Low % / High % + Apply**: adjust the contrast cutoff, click Apply, confirm the image
  actually changes.
- **Maximize**: click, confirm the figure grows to fill most of the screen; click
  again (now labeled "Restore") to shrink back.
- **Save GeoJSON**, then re-run the notebook from the top and confirm the saved core(s)
  reload correctly.